In [1]:
import json
import re
from collections import defaultdict

# List of all Eevee evolutions
EEVOLUTIONS = [
    "vaporeon", "jolteon", "flareon", "espeon",
    "umbreon", "leafeon", "glaceon", "sylveon"
]

def parse_pokedex(txt):
    pokedex      = {}
    wrong_format = []
    suffix_ctr   = defaultdict(int)   # counts per base name

    for raw_line in txt.splitlines():
        line = raw_line.strip()
        if not line:
            continue

        # Special: each Eevee evolution becomes its own 2‑stage chain
        m = re.match(r'^(eevee)\s+(\d+)\s*\(tutte le eevoluzioni\)', line, re.IGNORECASE)
        if m:
            lvl = int(m.group(2))
            for evo in EEVOLUTIONS:
                # assign a unique eevee name for each evo
                base_idx = suffix_ctr["eevee"]
                base_name = "eevee" if base_idx == 0 else f"eevee_{base_idx}"
                suffix_ctr["eevee"] += 1

                # also suffix the evo if it's already used
                evo_idx = suffix_ctr[evo]
                evo_name = evo if evo_idx == 0 else f"{evo}_{evo_idx}"
                suffix_ctr[evo] += 1

                pokedex[base_name] = ["base", [lvl, evo_name]]
                pokedex[evo_name]  = ["last", [lvl, base_name]]
            continue

        # --- otherwise, standard parsing ---
        tokens = line.split()
        names  = tokens[0::2]
        str_lv = tokens[1::2]

        try:
            lvls = [int(x) for x in str_lv]
        except ValueError:
            wrong_format.append(line)
            continue

        # 3‑stage chain
        if len(names) == 3 and len(lvls) == 2:
            base_o, mid_o, last_o = names
            lvl_mid, lvl_last     = lvls

            # assign unique keys
            name_map = {}
            for orig in (base_o, mid_o, last_o):
                idx = suffix_ctr[orig]
                new = orig if idx == 0 else f"{orig}_{idx}"
                name_map[orig] = new
                suffix_ctr[orig] += 1

            b, m_, l = name_map[base_o], name_map[mid_o], name_map[last_o]
            pokedex[b]  = ["base", [lvl_mid, m_], [lvl_last, l]]
            pokedex[m_] = ["mid",  [lvl_mid, b],  [lvl_last, l]]
            pokedex[l]  = ["last", [lvl_mid, b],  [lvl_last, m_]]

        # 2‑stage chain
        elif len(names) == 2 and 1 <= len(lvls) <= 2:
            base_o, last_o = names
            lvl_last       = lvls[-1]

            name_map = {}
            for orig in (base_o, last_o):
                idx = suffix_ctr[orig]
                new = orig if idx == 0 else f"{orig}_{idx}"
                name_map[orig] = new
                suffix_ctr[orig] += 1

            b, l = name_map[base_o], name_map[last_o]
            pokedex[b] = ["base", [lvl_last, l]]
            pokedex[l] = ["last", [lvl_last, b]]
        else:
            wrong_format.append(line)

    return pokedex, wrong_format

In [2]:
with open(r'D:\Documents\GitHub\MawileBot\home\utils\pokemon_evo_list\livelli_evo.txt', 'r', encoding='utf-8') as f:
    raw = f.read()

pokedex, wrong = parse_pokedex(raw)

# write out JSON
with open(r'D:\Documents\GitHub\MawileBot\home\utils\pokemon_evo_list\evo_file.json', 'w', encoding='utf-8') as f:
    json.dump(pokedex, f, indent=4, ensure_ascii=False)

print(f"Wrote pokedex.json with {len(pokedex)} entries.")
if wrong:
    print("\nThe following lines were in the wrong format and skipped:")
    for ln in wrong:
        print("  •", ln)

Wrote pokedex.json with 928 entries.
